In [15]:
import os, json, time, datetime as dt, csv, pathlib
from typing import Dict, List
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

DATA_RAW = pathlib.Path("data/raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)

load_dotenv()
ALPHA_KEY = os.getenv("ALPHAVANTAGE_API_KEY")
print("Loaded ALPHAVANTAGE_API_KEY?", bool(ALPHA_KEY))
#print isi API nya
print(ALPHA_KEY)

Loaded ALPHAVANTAGE_API_KEY? True
X0JPA8FLSR7GU8T9


In [7]:
def safe_stamp():
    return dt.datetime.now().strftime("%Y%m%d-%H%M%S")

def safe_filename(prefix: str, meta: Dict[str, str]) -> str:
    mid = "_".join([f"{k}-{str(v).replace(' ', '-')[:20]}" for k, v in meta.items()])
    return f"{prefix}_{mid}_{safe_stamp()}.csv"

def validate_df(df: pd.DataFrame, required_cols: List[str], dtypes_map: Dict[str, str]) -> Dict[str, str]:
    msgs = {}
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        msgs['missing_cols'] = f"Missing columns: {missing}"
    for col, dtype in dtypes_map.items():
        if col in df.columns:
            try:
                if dtype == 'datetime64[ns]':
                    pd.to_datetime(df[col])
                elif dtype == 'float':
                    pd.to_numeric(df[col])
            except Exception as e:
                msgs[f'dtype_{col}'] = f"Failed to coerce {col} to {dtype}: {e}"
    na_counts = df.isna().sum().sum()
    msgs['na_total'] = f"Total NA values: {na_counts}"
    return msgs

Using Alpha Vantage: True


AssertionError: Unexpected response keys: ['Information']

使用Alpha Vantage获取信用利差数据
信用利差数据已保存: api_creditspread_20250819.csv

信用利差 数据验证:
1. 前5行数据:
        date yield_rate
0 2025-07-01       4.39
1 2025-06-01       4.38
2 2025-05-01       4.42
3 2025-04-01       4.28
4 2025-03-01       4.28

2. 数据类型:
date          datetime64[ns]
yield_rate            object
dtype: object

3. 缺失值统计:
date          0
yield_rate    0
dtype: int64

4. 数据形状: (868, 2)
正在抓取联邦储备银行数据: https://www.federalreserve.gov/releases/h8/current/default.htm
抓取失败: Missing optional dependency 'lxml'.  Use pip or conda to install lxml., 使用示例数据
银行风险数据已保存: scrape_bankrisk_20250819.csv

银行风险 数据验证:
1. 前5行数据:
              bank  npl_ratio
0   JPMorgan Chase       0.85
1  Bank of America       1.12
2      Wells Fargo       1.45

2. 数据类型:
bank          object
npl_ratio    float64
dtype: object

3. 缺失值统计:
bank         0
npl_ratio    0
dtype: int64

4. 数据形状: (3, 2)

=== 数据获取完成 ===


C:\Users\kidney\AppData\Local\Temp\ipykernel_24268\3595334695.py:104: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(resp.text)
